# Evaluación de RAG: experimentos sobre colecciones de Qdrant

Este cuaderno barre configuraciones (colecciones, modelos, `top_k`, filtros)
sobre colecciones que ya existen en Qdrant, calcula las métricas de
`hermes_report.metrics` y guarda **una fila por (ejecución × colección)** en un
histórico que se puede comparar entre ejecuciones.

Cómo se usa:

1. Ejecuta las celdas en orden. Cada celda muestra su resultado intermedio.
2. Para otro experimento cambia solo el diccionario `CONFIG` (sección 2) y
   vuelve a ejecutar desde ahí.
3. Con `HERMES_NOTEBOOK_OFFLINE=1` todo corre sin Ollama ni Qdrant (embedder
   hash, LLM y juez de eco, Qdrant en memoria): sirve para ver que el cuaderno
   funciona, no para medir calidad.

El cuaderno hace su propia recuperación y generación (ADR 0003); solo lee las
colecciones que le indicas y solo escribe colecciones `hermes_report_*`.

## 1. Preparación

Importaciones, modo (con o sin conexión) y carpeta de salida. La carpeta de
salida es `HERMES_NOTEBOOK_OUT` o, si no existe, `notebooks/out/`.

In [ ]:
# Section 1: imports, mode and output folder.
import hashlib
import json
import os
import time
import uuid
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
from hermes_core import PromptMessage
from hermes_enrich.tokens import default_token_counter
from hermes_generator import GenerationParams
from IPython.display import display
from qdrant_client import models

from hermes_report.config import (
  EmbeddingConfig,
  GenerationConfig,
  IngestConfig,
  JudgeConfig,
)
from hermes_report.corpus import load_corpus
from hermes_report.ingest import ingest_corpus
from hermes_report.metrics import (
  JudgeSession,
  MetricInput,
  MetricSelection,
  MetricServices,
  RankedChunk,
  aggregate,
  evaluate_metrics,
)
from hermes_report.providers import (
  check_ollama,
  make_embedder,
  make_judge,
  make_llm,
)
from hermes_report.store import PREFIX, ensure_collection, make_client

# Offline mode replaces every service by a local stand-in (no Ollama, no Qdrant).
OFFLINE = os.environ.get("HERMES_NOTEBOOK_OFFLINE") == "1"


def find_root() -> Path:
  """The repository folder: the nearest parent of the cwd that holds the demo."""
  here = Path.cwd().resolve()
  for folder in (here, *here.parents):
    if (folder / "examples" / "demo" / "corpus").is_dir():
      return folder
  raise RuntimeError("Abre el cuaderno dentro del repositorio hermes-report-generator.")


ROOT = find_root()
OUT_DIR = Path(os.environ.get("HERMES_NOTEBOOK_OUT") or ROOT / "notebooks" / "out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

display(
  {"modo": "sin conexión" if OFFLINE else "con Ollama y Qdrant", "salida": str(OUT_DIR)}
)

## 2. Configuración del experimento

Todo lo que cambia entre experimentos está en `CONFIG`:

| Clave | Qué es |
|---|---|
| `collections` | colecciones de Qdrant a evaluar (una fila de resultados por colección) |
| `qdrant_url` | URL de Qdrant (se ignora sin conexión) |
| `embedding`, `generation`, `judge` | proveedor y modelo de cada pieza; `judge.provider = None` desactiva el juez |
| `top_k`, `score_threshold` | cuántos fragmentos se recuperan y con qué puntuación mínima |
| `filters` | condiciones sobre el payload, p. ej. `{"source": "manuales/faro.md"}` (una lista significa «cualquiera de») |
| `metrics` | familias activadas por defecto más `include` menos `exclude` |
| `dataset` | JSON de preguntas (lista, o `{"title", "questions"}`) |
| `prompt.insufficient_message` | respuesta cuando las fuentes no bastan |
| `note` | texto libre que queda en el histórico |

Variables de entorno opcionales (para ejecutar sin abrir el cuaderno):
`HERMES_NOTEBOOK_TOP_K`, `HERMES_NOTEBOOK_COLLECTIONS` (separadas por comas),
`HERMES_NOTEBOOK_DATASET` y `HERMES_NOTEBOOK_QDRANT_URL`.

In [ ]:
# Section 2: the one place to edit between experiments.
CONFIG: dict[str, Any] = {
  # Existing collections to evaluate; `hermes-report run` prints the name of its own.
  "collections": ["hermes_report_don_quijote"],
  "qdrant_url": "http://localhost:6333",
  "embedding": {"provider": "ollama", "model": "nomic-embed-text"},
  "generation": {
    "provider": "ollama",
    "model": "qwen3:4b",
    "temperature": 0.0,
    "seed": 42,
  },
  # provider None switches the judge (and every judged metric) off.
  "judge": {"provider": "ollama", "model": "qwen3:4b"},
  "top_k": 10,
  "score_threshold": None,
  "filters": {},
  "metrics": {"enabled": True, "include": [], "exclude": []},
  "dataset": "examples/quijote/preguntas.json",
  "prompt": {"insufficient_message": "No encuentro esa información en las fuentes."},
  "note": "",
}

# Optional overrides from the environment (so a run needs no notebook edit).
if os.environ.get("HERMES_NOTEBOOK_TOP_K"):
  CONFIG["top_k"] = int(os.environ["HERMES_NOTEBOOK_TOP_K"])
if os.environ.get("HERMES_NOTEBOOK_COLLECTIONS"):
  names = os.environ["HERMES_NOTEBOOK_COLLECTIONS"].split(",")
  CONFIG["collections"] = [name.strip() for name in names if name.strip()]
if os.environ.get("HERMES_NOTEBOOK_DATASET"):
  CONFIG["dataset"] = os.environ["HERMES_NOTEBOOK_DATASET"]
if os.environ.get("HERMES_NOTEBOOK_QDRANT_URL"):
  CONFIG["qdrant_url"] = os.environ["HERMES_NOTEBOOK_QDRANT_URL"]

# Offline: hash embedder, echo LLM and judge, the demo collection and the demo questions.
DEMO_COLLECTION = PREFIX + "notebook_demo"
if OFFLINE:
  CONFIG["collections"] = [DEMO_COLLECTION]
  CONFIG["qdrant_url"] = ":memory:"
  CONFIG["embedding"] = {"provider": "hash", "model": "hash-256"}
  CONFIG["generation"] = {**CONFIG["generation"], "provider": "echo", "model": "echo"}
  CONFIG["judge"] = {"provider": "echo", "model": "echo"}
  CONFIG["dataset"] = "examples/demo/preguntas.json"

# Identifies the configuration: equal configs share the hash, so rows are comparable.
CONFIG_HASH = hashlib.sha256(
  json.dumps(CONFIG, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
).hexdigest()[:12]

display(CONFIG)
display({"config_hash": CONFIG_HASH})

In [ ]:
# Section 2 (cont.): the clients the configuration asks for.
# Ollama is checked first so a missing model fails here, not in the middle of a sweep.
if not OFFLINE:
  wanted = {
    "embedding": CONFIG["embedding"],
    "generation": CONFIG["generation"],
    "judge": CONFIG["judge"],
  }
  models_needed = sorted(
    {part["model"] for part in wanted.values() if part.get("provider") == "ollama"}
  )
  if models_needed:
    check_ollama(models_needed)

client = make_client(CONFIG["qdrant_url"])
embedder = make_embedder(EmbeddingConfig(**CONFIG["embedding"]))
llm = make_llm(
  GenerationConfig(
    **CONFIG["generation"],
    insufficient_message=CONFIG["prompt"]["insufficient_message"],
  )
)
judge_client = (
  make_judge(JudgeConfig(enabled=True, **CONFIG["judge"]))
  if CONFIG["judge"].get("provider")
  else None
)
counter = default_token_counter()

# Reading is free: list what Qdrant holds so a wrong collection name is easy to spot.
available = sorted(c.name for c in client.get_collections().collections)
display({"colecciones en Qdrant": available})

## 3. Colección sin conexión

Solo se ejecuta con `HERMES_NOTEBOOK_OFFLINE=1`: ingiere
`examples/demo/corpus` con las funciones públicas de `hermes_report` y crea la
colección `hermes_report_notebook_demo` en el Qdrant en memoria. Con conexión
no se crea ni se escribe ninguna colección.

In [ ]:
# Section 3: build the demo collection (offline only). `ensure_collection` refuses
# any name that does not start with "hermes_report_", so nothing else can be written.
if OFFLINE:
  corpus = load_corpus(ROOT / "examples" / "demo" / "corpus")
  ingestion = ingest_corpus(
    corpus,
    IngestConfig(max_chunk_size=300),
    EmbeddingConfig(**CONFIG["embedding"]),
    embedder,
    OUT_DIR / "ingest-cache",
  )
  info = ensure_collection(client, DEMO_COLLECTION, ingestion.points, ":memory:")
  display(info)
else:
  display("Con conexión: se usan las colecciones indicadas en CONFIG, solo en lectura.")

## 4. Preguntas

`load_golden` acepta una lista JSON o un objeto `{"title", "questions": [...]}`.
`ground_truth` es un alias de `expected` y el `id` por defecto es `q01`, `q02`…
Los campos `documents`, `evidence` y `refusal` son opcionales: sin ellos las
métricas que los necesitan quedan como «no aplicable».

In [ ]:
# Section 4: load the questions with their expected answers.
def load_golden(path: str | Path) -> pd.DataFrame:
  """Read a questions file (array, or object with "questions") into a table."""
  file = Path(path)
  if not file.is_absolute() and (ROOT / file).exists():
    file = ROOT / file
  raw = json.loads(file.read_text(encoding="utf-8"))
  items = raw["questions"] if isinstance(raw, dict) else raw
  rows = []
  for number, item in enumerate(items, start=1):
    question = item.get("question")
    if not isinstance(question, str) or not question.strip():
      raise ValueError(f"{file.name}: la pregunta {number} no tiene texto")
    expected = item.get("expected", item.get("ground_truth", ""))
    rows.append(
      {
        "id": str(item.get("id") or f"q{number:02d}"),
        "question": question,
        "expected": str(expected or ""),
        "documents": list(item.get("documents") or []),
        "evidence": list(item.get("evidence") or []),
        "refusal": bool(item.get("refusal", False)),
      }
    )
  columns = ["id", "question", "expected", "documents", "evidence", "refusal"]
  return pd.DataFrame(rows, columns=columns)  # an empty file keeps its columns


golden = load_golden(CONFIG["dataset"])
if golden.empty:
  print("El conjunto de preguntas está vacío: no habrá métricas por pregunta.")
display(golden)

## 5. Recuperar y generar

Para cada colección y cada pregunta: embeber, buscar en Qdrant con `top_k`,
`score_threshold` y los filtros, construir el prompt numerado, llamar al LLM y
detectar si rechazó contestar. El contexto son los fragmentos recuperados en
orden (este cuaderno no enriquece el contexto). Cada resultado se guarda como
un `MetricInput`.

In [ ]:
# Section 5: retrieval and generation, done here on purpose (see the intro).
SENTINEL = (
  "INSUFFICIENT_CONTEXT"  # what the system prompt asks for when sources fall short
)
SYSTEM = (
  "Eres un asistente que responde solo con las fuentes dadas. Cita cada "
  "afirmación con el número de su fuente entre corchetes, por ejemplo [1]. "
  f"Si las fuentes no bastan para responder, responde exactamente {SENTINEL}."
)
INSUFFICIENT = CONFIG["prompt"]["insufficient_message"]


def build_filter(filters: dict[str, Any]) -> models.Filter | None:
  """Payload conditions: a value must match, a list means any of its values."""
  if not filters:
    return None
  must = []
  for key, value in filters.items():
    if isinstance(value, list | tuple):
      match: models.MatchAny | models.MatchValue = models.MatchAny(any=list(value))
    else:
      match = models.MatchValue(value=value)
    must.append(models.FieldCondition(key=key, match=match))
  return models.Filter(must=must)


def build_prompt(
  question: str, chunks: tuple[RankedChunk, ...]
) -> tuple[PromptMessage, ...]:
  """The numbered prompt: one <source n="N"> block per retrieved chunk."""
  blocks = []
  for number, chunk in enumerate(chunks, start=1):
    doc = (chunk.source or chunk.chunk_id).replace('"', "'")
    blocks.append(f'<source n="{number}" doc="{doc}">\n{chunk.text}\n</source>')
  user = f"Pregunta: {question}\n\nFuentes:\n" + "\n".join(blocks)
  return (
    PromptMessage(role="system", content=SYSTEM),
    PromptMessage(role="user", content=user),
  )


def answer_one(collection: str, row: Any) -> tuple[MetricInput, str | None]:
  """Retrieve and generate for one question; a failure keeps what was done."""
  retrieved: tuple[RankedChunk, ...] = ()
  prompt: tuple[PromptMessage, ...] = ()
  answer: str | None = None
  refused: bool | None = None
  replies: tuple[str, ...] = ()
  seconds: dict[str, float] = {}
  error: str | None = None
  try:
    begin = time.perf_counter()
    vector = embedder.get_query_embedding(row.question)
    response = client.query_points(
      collection_name=collection,
      query=vector,
      limit=CONFIG["top_k"],
      score_threshold=CONFIG["score_threshold"],
      query_filter=build_filter(CONFIG["filters"]),
      with_payload=True,
    )
    retrieved = tuple(
      RankedChunk(
        str(hit.id),
        str((hit.payload or {}).get("text", "")),
        (hit.payload or {}).get("source"),
        float(hit.score),
      )
      for hit in response.points
    )
    seconds["retrieval"] = time.perf_counter() - begin
    prompt = build_prompt(row.question, retrieved)
    begin = time.perf_counter()
    reply = llm.complete(
      prompt,
      GenerationParams(
        temperature=CONFIG["generation"]["temperature"],
        seed=CONFIG["generation"]["seed"],
      ),
    )
    seconds["generation"] = time.perf_counter() - begin
    replies = (reply.text,)
    refused = reply.text.strip().upper().startswith(SENTINEL)
    answer = INSUFFICIENT if refused else reply.text.strip()
  except Exception as exc:
    error = f"{type(exc).__name__}: {exc}"
  item = MetricInput(
    question_id=row.id,
    question=row.question,
    expected=row.expected,
    refusal=row.refusal,
    answer=answer,
    refused=refused,
    retrieved=retrieved,
    context=tuple(chunk.text for chunk in retrieved),
    documents=tuple(row.documents),
    evidence=tuple(row.evidence),
    seconds=seconds,
    prompt=prompt,
    replies=replies,
    citations=None,
  )
  return item, error


items: dict[str, list[MetricInput]] = {}
pipeline_errors: dict[tuple[str, str], str] = {}
for collection in CONFIG["collections"]:
  items[collection] = []
  for row in golden.itertuples():
    item, error = answer_one(collection, row)
    items[collection].append(item)
    if error:
      pipeline_errors[(collection, item.question_id)] = error

generated = pd.DataFrame(
  [
    {
      "collection": collection,
      "question_id": item.question_id,
      "retrieved": len(item.retrieved),
      "refused": item.refused,
      "answer": item.answer,
      "error": pipeline_errors.get((collection, item.question_id)),
    }
    for collection, collection_items in items.items()
    for item in collection_items
  ],
  columns=["collection", "question_id", "retrieved", "refused", "answer", "error"],
)
display(generated)

In [ ]:
# Section 5 (cont.): one complete example, to read what the metrics will see.
example_items = next((found for found in items.values() if found), [])
if example_items:
  example = example_items[0]
  display(
    {
      "pregunta": example.question,
      "esperada": example.expected,
      "respuesta": example.answer,
      "fragmentos": [(c.source, round(c.score, 3)) for c in example.retrieved],
    }
  )
elif not CONFIG["collections"]:
  print("No hay colecciones en CONFIG['collections']: no hay ejemplo que mostrar.")
else:
  print("No hay preguntas: no hay ejemplo que mostrar.")

## 6. Métricas

`evaluate_metrics` calcula cada familia seleccionada para cada pregunta, con
una `JudgeSession` nueva por pregunta (el juez no repite llamadas dentro de una
pregunta). Una métrica que falla no detiene las demás: queda con estado
`error` y su mensaje. Sin juez, las familias que lo necesitan quedan como
`skipped`.

In [ ]:
# Section 6: metrics per question, one column per score.
selection = CONFIG["metrics"]
selected = MetricSelection(
  selection["enabled"], tuple(selection["include"]), tuple(selection["exclude"])
).resolve()
score_names = [score for metric in selected for score in metric.scores]

results: dict[str, list[tuple[Any, ...]]] = {}
metric_rows = []
for collection, collection_items in items.items():
  results[collection] = []
  for item in collection_items:
    services = MetricServices(
      judge=None if judge_client is None else JudgeSession(judge_client, item),
      embedder=embedder,
      counter=counter,
    )
    outcome = evaluate_metrics(item, selected, services)
    results[collection].append(outcome)
    row: dict[str, Any] = {"collection": collection, "question_id": item.question_id}
    for result in outcome:
      row.update(result.scores)
      row[f"{result.name}_status"] = result.status
      row[f"{result.name}_error"] = result.error
    metric_rows.append(row)

per_question = pd.DataFrame(metric_rows)
if per_question.empty:  # keep the key columns so the detail can still be merged
  per_question = pd.DataFrame(columns=["collection", "question_id"])
display(per_question)

In [ ]:
# Section 6 (cont.): what did not go well, so a missing number is never a mystery.
status_columns = [c for c in per_question.columns if c.endswith("_status")]
problems = []
for column in status_columns:
  family = column.removesuffix("_status")
  bad = per_question[~per_question[column].isin(["ok", "not_applicable"])]
  for _, line in bad.iterrows():
    problems.append(
      {
        "collection": line["collection"],
        "question_id": line["question_id"],
        "family": family,
        "status": line[column],
        "error": line[f"{family}_error"],
      }
    )
display(
  pd.DataFrame(
    problems, columns=["collection", "question_id", "family", "status", "error"]
  )
)

## 7. Fila del experimento

Una fila por colección: la configuración usada, las medias de cada puntuación
(`<puntuación>_mean`) y cuántas preguntas fallaron (`<puntuación>_errors`).
`config_hash` identifica la configuración: dos filas con el mismo hash son
repeticiones; con distinto hash y la misma colección, un cambio que se puede
comparar.

In [ ]:
# Section 7: one experiment row per collection.
RUN_ID = datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:4]
TIMESTAMP = datetime.now().astimezone().isoformat(timespec="seconds")

experiment_rows = []
for collection, collection_items in items.items():
  summary = aggregate(results[collection], selected)
  total_seconds = [sum(item.seconds.values()) for item in collection_items]
  row = {
    "timestamp": TIMESTAMP,
    "run_id": RUN_ID,
    "collection": collection,
    "embedding_model": CONFIG["embedding"]["model"],
    "llm_model": CONFIG["generation"]["model"],
    "judge_model": CONFIG["judge"].get("model") if judge_client else None,
    "top_k": CONFIG["top_k"],
    "score_threshold": CONFIG["score_threshold"],
    "filters": json.dumps(CONFIG["filters"], sort_keys=True, ensure_ascii=False),
    "n_questions": len(collection_items),
  }
  for name in score_names:
    row[f"{name}_mean"] = summary[name].mean
    row[f"{name}_errors"] = summary[name].errors
  row["seconds_mean"] = (
    sum(total_seconds) / len(total_seconds) if total_seconds else None
  )
  row["config_hash"] = CONFIG_HASH
  row["note"] = CONFIG["note"]
  experiment_rows.append(row)

experiment = pd.DataFrame(experiment_rows)
display(experiment.T)
if experiment.empty:
  print("No hay colecciones que evaluar: no hay fila de experimento.")

## 8. Histórico

Cada ejecución añade sus filas a `history.csv` (si aparecen columnas nuevas, las
filas antiguas quedan con el valor vacío) y a `history.jsonl`, y escribe el
detalle por pregunta en `<run_id>-detail.csv`. Después se recarga el histórico y
se ordena por la puntuación elegida en `SORT_BY`. Sin preguntas, cada colección
escribe igualmente su fila (`n_questions` = 0, medias vacías); sin colecciones no
se escribe ninguna fila.

In [ ]:
# Section 8: append to the history, write the detail, reload and sort.
HISTORY_CSV = OUT_DIR / "history.csv"
HISTORY_JSONL = OUT_DIR / "history.jsonl"
DETAIL_CSV = OUT_DIR / f"{RUN_ID}-detail.csv"
SORT_BY = "correctness_f1_mean"  # any "<score>_mean" column of the history

if not experiment_rows:
  # No collection, no row: nothing is written, so the history stays as it was.
  print("No hay colecciones que evaluar: no se escribió ninguna fila en el histórico.")
else:
  # CSV: union of the columns of the old rows and the new ones.
  previous = pd.read_csv(HISTORY_CSV) if HISTORY_CSV.exists() else pd.DataFrame()
  pd.concat([previous, experiment], ignore_index=True).to_csv(HISTORY_CSV, index=False)

  # JSON lines: one object per row, so a later column never rewrites the old lines.
  with HISTORY_JSONL.open("a", encoding="utf-8") as handle:
    for row in experiment_rows:
      handle.write(json.dumps(row, ensure_ascii=False) + "\n")

  # Detail: the scores of every question plus what was answered.
  detail = per_question.merge(generated, on=["collection", "question_id"], how="left")
  detail.insert(0, "run_id", RUN_ID)
  detail.to_csv(DETAIL_CSV, index=False)

  history = pd.read_csv(HISTORY_CSV)
  sort_by = SORT_BY if SORT_BY in history.columns else "seconds_mean"
  shown = [
    "run_id",
    "collection",
    "top_k",
    "n_questions",
    sort_by,
    "config_hash",
    "note",
  ]
  display(
    history.sort_values(sort_by, ascending=False, na_position="last")[
      list(dict.fromkeys(shown))
    ]
  )
  display({"histórico": str(HISTORY_CSV), "detalle": str(DETAIL_CSV)})